# T4 Energy Calibration for ShadowKV++

This notebook measures actual GPU energy (NVML) on a **Google Colab T4 GPU**
for a representative subset of the ShadowKV++ controlled benchmark matrix.
The measured J/req values are then extrapolated to the full 898-file result bundle.

**What you need:**
- The file `t4_energy_transfer.tgz` (download from wherever it's stored)
- A Colab runtime with T4 GPU (Runtime → Change runtime type → T4 GPU)

**Timing:** ~40 min (quick, 20 cells) or ~72 min (full, 36 cells)

## 1. Upload the Package

Click the button below and select `t4_energy_transfer.tgz` from your computer.

In [ ]:
from google.colab import files
import tarfile, os, shutil, sys, time, zipfile, glob, json
import pandas as pd

print('Please select t4_energy_transfer.tgz...')
uploaded = files.upload()

# Find the tgz in the uploaded files
tgz_path = None
for fname in uploaded.keys():
    if fname.endswith('.tgz') or fname.endswith('.tar.gz'):
        tgz_path = fname
        break

if tgz_path is None:
    raise RuntimeError('No .tgz file found in upload. Please upload t4_energy_transfer.tgz')

print(f'Extracting {tgz_path}...')
with tarfile.open(tgz_path, 'r:gz') as tar:
    tar.extractall('/content/')

# Find the extracted directory
if os.path.exists('/content/t4_energy_transfer'):
    os.chdir('/content/t4_energy_transfer')
    print(f'Extracted to /content/t4_energy_transfer')
else:
    # Maybe there's a single top-level dir inside
    contents = [d for d in os.listdir('/content') if os.path.isdir(f'/content/{d}') and d != 'sample_data']
    if contents:
        os.chdir(f'/content/{contents[0]}')
        print(f'Extracted to /content/{contents[0]}')
    else:
        raise RuntimeError('Could not find extracted directory')

print(f'Working directory: {os.getcwd()}')
print(f'Files: {len(os.listdir("."))}')

## 2. Install Dependencies (~3-5 min)

Installs the ShadowKV++ package and its Python dependencies.

In [ ]:
print('Installing dependencies...')
!pip install -U pip -q
!pip install -e . -q
!pip install pytest -q

# Verify GPU
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print(f'\nGPU: {gpu_name}')
if torch.cuda.is_available():
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
    print(f'CUDA: {torch.version.cuda}')
else:
    print('WARNING: No GPU detected. Change runtime type to T4 GPU.')

## 3. Smoke Test (CPU, ~10 seconds)

Verifies the installation is working before spending GPU time.

In [ ]:
print('Running smoke test...')
!python experiments/run_benchmark.py --backend fake --workload synthetic \
  --n_requests 8 --include_experimental --disable_arrival_simulation \
  --output_dir /tmp/smoke
print('\nSmoke test passed! Environment is ready.')

## 4. Run Energy Calibration Experiment

This is the main experiment. It runs the benchmark with `--measure_energy`
on a small matrix to get actual GPU energy readings.

In [ ]:
# ── Choose mode ─────────────────────────────────────────────────────
# Set QUICK_MODE = True for 20 cells (~40 min, 5 engines)
# Set QUICK_MODE = False for 36 cells (~72 min, all 9 engines)
QUICK_MODE = False

label = 'quick' if QUICK_MODE else 'full'
n_cells = 20 if QUICK_MODE else 36
est_min = 40 if QUICK_MODE else 72

print(f'Mode: {"QUICK" if QUICK_MODE else "FULL"}')
print(f'Cells: {n_cells}')
print(f'Estimated time: ~{est_min} min')
print()
print('Engines:')
if QUICK_MODE:
    print('  no_cache, reactive_prefix_cache, greedy_prefix_cache, shadow_kv, shadow_kv_plus')
else:
    print('  no_cache, reactive, greedy, strict_reactive, frequency_speculative,')
    print('  shadow_kv, shadow_kv_plus, shadow_kv_plus_best_latency, shadow_kv_plus_raw_observer')
print()
r = input('Ready to start? (yes/no): ')
if r.lower() != 'yes':
    print('Cancelled. Run this cell again when ready.')
    raise SystemExit()

In [ ]:
start_time = time.time()

cmd = [sys.executable, 'experiments/run_t4_energy_calibration.py']
if QUICK_MODE:
    cmd.append('--quick')

print(f'Starting: {" ".join(cmd)}')
print('=' * 60)

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR (last 2000 chars):')
    print(result.stderr[-2000:])

elapsed_min = (time.time() - start_time) / 60
print(f'\nExperiment finished in {elapsed_min:.0f} minutes.')

## 5. Extrapolate to Full Result Bundle

Uses the measured J/req to estimate GPU energy for all 898 controlled-result
JSONs (T4 + P100, 5 models, 10 datasets, 3 modes, 3 seeds).

In [ ]:
print('Running calibration...')
!python experiments/calibrate_energy_from_measurement.py
print('\nCalibration complete.')

## 6. Download Results

The results are zipped and a download link appears below. Save it to your computer.

In [ ]:
# Zip all results
results_dir = f'/content/t4_energy_transfer/results'
zip_path = f'/content/t4_energy_calibration_{label}.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(results_dir):
        for f in files:
            file_path = os.path.join(root, f)
            arcname = os.path.relpath(file_path, results_dir)
            zf.write(file_path, arcname)

print(f'Results zipped: {zip_path}')
print(f'Size: {os.path.getsize(zip_path) / 1024 / 1024:.0f} MB')
print()
print('Contents:')
with zipfile.ZipFile(zip_path, 'r') as zf:
    for info in zf.infolist():
        print(f'  {info.filename} ({info.file_size} bytes)')

In [ ]:
from google.colab import files as colab_files
colab_files.download(zip_path)

## 7. Preview: Headline Energy Estimates

Aggregate energy per engine across the full controlled bundle.

In [ ]:
summary_path = '/content/t4_energy_transfer/results/t4_energy_calibration/summary_by_engine_energy.csv'
if os.path.exists(summary_path):
    df = pd.read_csv(summary_path)
    print('Headline Energy Estimates (T4 + P100 combined):')
    print()
    print(df.to_string(index=False))
else:
    print('Summary CSV not found yet. The calibration step may still be running.')

## Notes

- **Session timeout**: Colab may disconnect if idle. The runner saves progress
  to `_run_manifest.json` after each cell, so you can resume.
- **Model**: Qwen2.5-1.5B-Instruct (~3 GB) is downloaded from HuggingFace.
  Colab caches it for the session duration.
- **Energy measurement**: Uses NVML via `pynvml`. Works on any NVIDIA GPU.
  The T4 in Colab supports NVML energy readings.
- **Results**: All output saved to `/content/t4_energy_transfer/results/`.
  Download the zip from cell 6 before the session ends.
- **Failed cells**: If some benchmark cells fail, the runner continues with
  the remaining cells. Check `_run_manifest.json` for failure details.